# Heating Oil COT Mapping: CFTC → Marouen's cot_db

Goal: reconstruct the HO rows in `cot_db.csv` from public CFTC data.

Heating Oil is NYMEX-only (code `022651`), no ICE leg.
- Commercial = Disaggregated Prod/Merch (Combined)
- ManagedMoney = Legacy Non-Commercial (Combined)

In [1]:
import pandas as pd

disagg = pd.read_csv("../downloads/cftc/disaggregated_combined.csv", low_memory=False)
legacy = pd.read_csv("../downloads/cftc/legacy_combined.csv", low_memory=False)
marouen = pd.read_csv("../cot_data/cot_db.csv")

## Step 1 — Find Heating Oil in CFTC files

In [2]:
code = "022651"

for label, df in [("Disaggregated", disagg), ("Legacy", legacy)]:
    sub = df[df["cftc_contract_market_code"] == code]
    print(f"\n{label} — Code {code}:")
    print("Market and Exchange Names:")
    for n in sub["market_and_exchange_names"].unique():
        print(f"  {n}")
    if "commodity_name" in df.columns:
        print("Commodity Names:")
        for c in sub["commodity_name"].unique():
            print(f"  {c}")
    if "contract_market_name" in df.columns:
        print("Contract Market Names:")
        for cm in sub["contract_market_name"].unique():
            print(f"  {cm}")


Disaggregated — Code 022651:
Market and Exchange Names:
  NO. 2 HEATING OIL, N.Y. HARBOR - NEW YORK MERCANTILE EXCHANGE
  #2 HEATING OIL, NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE
  #2 HEATING OIL- NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE
  NY HARBOR USLD - NEW YORK MERCANTILE EXCHANGE
  NY HARBOR ULSD - NEW YORK MERCANTILE EXCHANGE
Commodity Names:
  HEATING OIL-DIESEL-GASOIL
Contract Market Names:
  NY HARBOR ULSD

Legacy — Code 022651:
Market and Exchange Names:
  NO. 2 HEATING OIL, N.Y. HARBOR - NYME HEATING OIL #2
  NO. 2 HEATING OIL, N.Y. HARBOR - NEW YORK MERCANTILE EXCHANGE
  #2 HEATING OIL, NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE
  #2 HEATING OIL- NY HARBOR-ULSD - NEW YORK MERCANTILE EXCHANGE
  NY HARBOR USLD - NEW YORK MERCANTILE EXCHANGE
  NY HARBOR ULSD - NEW YORK MERCANTILE EXCHANGE
Commodity Names:
  HEATING OIL-DIESEL-GASOIL
Contract Market Names:
  NY HARBOR ULSD


## Step 2 — Build Commercial and ManagedMoney

In [3]:
# Commercial from disaggregated combined
ho_disagg = disagg[disagg["cftc_contract_market_code"] == code].copy()
ho_disagg["date"] = pd.to_datetime(ho_disagg["report_date_as_yyyy_mm_dd"])
ho_disagg["prod_merc_positions_long"] = pd.to_numeric(ho_disagg["prod_merc_positions_long"], errors="coerce")
ho_disagg["prod_merc_positions_short"] = pd.to_numeric(ho_disagg["prod_merc_positions_short"], errors="coerce")

# ManagedMoney from legacy combined
ho_legacy = legacy[legacy["cftc_contract_market_code"] == code].copy()
ho_legacy["date"] = pd.to_datetime(ho_legacy["report_date_as_yyyy_mm_dd"])
ho_legacy["noncomm_positions_long_all"] = pd.to_numeric(ho_legacy["noncomm_positions_long_all"], errors="coerce")
ho_legacy["noncomm_positions_short_all"] = pd.to_numeric(ho_legacy["noncomm_positions_short_all"], errors="coerce")

# Merge
rebuilt = pd.merge(
    ho_disagg[["date", "prod_merc_positions_long", "prod_merc_positions_short"]],
    ho_legacy[["date", "noncomm_positions_long_all", "noncomm_positions_short_all"]],
    on="date",
)
rebuilt.columns = ["date", "CommercialLong", "CommercialShort", "MMLong", "MMShort"]
rebuilt["CommercialNet"] = rebuilt["CommercialLong"] - rebuilt["CommercialShort"]
rebuilt["MMNet"] = rebuilt["MMLong"] - rebuilt["MMShort"]

print(f"Rebuilt HO: {len(rebuilt)} rows")
rebuilt.head()

Rebuilt HO: 1033 rows


,date,CommercialLong,CommercialShort,MMLong,MMShort,CommercialNet,MMNet
0,2006-06-13,56814,105948,20025,16509,-49134,3516
1,2006-06-20,57307,106497,20089,15866,-49190,4223
2,2006-06-27,52876,109722,24117,12466,-56846,11651
3,2006-07-03,51129,108590,25929,13643,-57461,12286
4,2006-07-11,53195,108382,22787,13077,-55187,9710


## Step 3 — Compare against Marouen's cot_db

In [4]:
marouen["tradeDate"] = pd.to_datetime(marouen["tradeDate"])
ho = marouen[marouen["Name"] == "HO"].dropna(subset=["Commercial_NetPosition"]).copy()

comp = pd.merge(rebuilt, ho, left_on="date", right_on="tradeDate", how="inner")
print(f"Matched rows: {len(comp)} (Marouen has {len(ho)})\n")

pairs = [
    ("CommercialLong",  "CommercialLongPosition"),
    ("CommercialShort", "CommercialShortPosition"),
    ("CommercialNet",   "Commercial_NetPosition"),
    ("MMLong",          "ManagedMoney_LongPosition"),
    ("MMShort",         "ManagedMoney_ShortPosition"),
    ("MMNet",           "ManagedMoney_NetPosition"),
]

print(f"{'rebuilt':20s}   {'marouen':30s}   {'MAPE':>8s}   {'MAE':>8s}   {'max_diff':>8s}")
print("-" * 90)
for ours, theirs in pairs:
    abs_diff = (comp[ours] - comp[theirs]).abs()
    mae = abs_diff.mean()
    denom = comp[theirs].abs().replace(0, float("nan"))
    mape = (abs_diff / denom * 100).mean()
    print(f"{ours:20s}   {theirs:30s}   {mape:7.4f}%   {mae:8.2f}   {abs_diff.max():.0f}")

Matched rows: 802 (Marouen has 809)

rebuilt                marouen                              MAPE        MAE   max_diff
------------------------------------------------------------------------------------------
CommercialLong         CommercialLongPosition            0.0000%       0.00   0
CommercialShort        CommercialShortPosition           0.0000%       0.00   0
CommercialNet          Commercial_NetPosition            0.0000%       0.00   0
MMLong                 ManagedMoney_LongPosition         0.0004%       0.20   1
MMShort                ManagedMoney_ShortPosition        0.0003%       0.16   1
MMNet                  ManagedMoney_NetPosition          0.0091%       0.33   2
